In [ ]:
# =========================================================
# INSTALL (Run once in Colab)
# =========================================================

!pip -q install rouge-score sacrebleu bert-score sentence-transformers

# =========================================================
# IMPORTS
# =========================================================

import json, ast, re, numpy as np
from collections import defaultdict

from rouge_score import rouge_scorer
import sacrebleu
from bert_score import score as bertscore

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# =========================================================
# FILE PATHS
# =========================================================

GT_PATH   = "/content/Testing_Ground_Truth_Data.txt"
PRED_PATH = "/content/final_test_predictions.json"

# =========================================================
# LOAD DATA
# =========================================================

def load_gt(path):
    with open(path,"r",encoding="utf-8") as f:
        txt = f.read()

    try:
        return json.loads(txt)
    except:
        return ast.literal_eval(txt)


def load_json(path):
    with open(path,"r",encoding="utf-8") as f:
        return json.load(f)


gt_data = load_gt(GT_PATH)
pred_data = load_json(PRED_PATH)

print("GT:",len(gt_data))
print("Pred:",len(pred_data))


# =========================================================
# ROBUST JSON EXTRACTION
# =========================================================

def extract_json(text):

    if text is None:
        return None

    text = str(text).replace("```json","").replace("```","").strip()

    # direct parse
    try:
        return json.loads(text)
    except:
        pass

    # array extraction
    candidates = re.findall(r"\[.*\]|\{.*\}", text, re.DOTALL)

    for c in reversed(candidates):
        try:
            return json.loads(c)
        except:
            continue

    return None


# =========================================================
# NORMALIZE SCENES
# =========================================================

def normalize(data):

    if isinstance(data,dict):

        if "output" in data:
            data=data["output"]

        elif "scenes" in data:
            data=data["scenes"]

    if isinstance(data,dict):
        data=[data]

    if not isinstance(data,list):
        return []

    scenes=[]

    for s in data:

        if not isinstance(s,dict):
            continue

        scenes.append({
            "emotion":str(s.get("emotion","")).lower().strip(),
            "setting":str(s.get("setting","")).lower().strip(),
            "atmosphere":str(s.get("atmosphere","")).lower().strip(),
            "state":str(s.get("state","")).lower().strip(),
            "theme":str(s.get("theme","")).lower().strip(),
            "state_change":str(s.get("state_change","")).lower().strip()
        })

    return scenes


# =========================================================
# PRF1
# =========================================================

def prf(gt_list,pred_list):

    gt=set([x for x in gt_list if x!=""])
    pr=set([x for x in pred_list if x!=""])

    tp=len(gt & pr)
    fp=len(pr-gt)
    fn=len(gt-pr)

    p=tp/(tp+fp+1e-9)
    r=tp/(tp+fn+1e-9)
    f=2*p*r/(p+r+1e-9)

    return p,r,f


def attr_metrics(gt,pred,key):

    return prf(
        [x[key] for x in gt],
        [x[key] for x in pred]
    )


# =========================================================
# STRUCTURAL METRICS
# =========================================================

def scene_acc(gt,pred):
    return 1-abs(len(gt)-len(pred))/max(len(gt),1)


def state_transition(gt,pred):

    gt_t=[x["state_change"] for x in gt if x["state_change"]]
    pr_t=[x["state_change"] for x in pred if x["state_change"]]

    return prf(gt_t,pr_t)[2]


def causal(gt,pred):

    a=[x["state"] for x in gt]
    b=[x["state"] for x in pred]

    n=min(len(a),len(b))

    if n==0:
        return 0

    return sum(a[i]==b[i] for i in range(n))/n


def schema(pred):

    req=["emotion","setting","atmosphere","state","theme"]

    total=0
    valid=0

    for s in pred:
        for r in req:
            total+=1
            if s.get(r,"")!="":
                valid+=1

    return valid/(total+1e-9)


# =========================================================
# EVALUATION LOOP
# =========================================================

results=defaultdict(list)

valid_json=0
exact_match=0
struct_match=0

refs=[]
hyps=[]

scene_counts=[]

for gt_story,pred_story in zip(gt_data,pred_data):

    gt_scenes=normalize(gt_story)

    pred_json=extract_json(
        pred_story["predicted_output"]
    )

    if pred_json is None:
        continue

    valid_json+=1

    pred_scenes=normalize(pred_json)

    if len(pred_scenes)==0:
        continue

    scene_counts.append(len(pred_scenes))

    # structural exact match
    if gt_scenes==pred_scenes:
        exact_match+=1

    if len(gt_scenes)==len(pred_scenes):
        struct_match+=1

    # references for text metrics
    refs.append(json.dumps(gt_scenes))
    hyps.append(json.dumps(pred_scenes))

    # scene
    results["scene_acc"].append(
        scene_acc(gt_scenes,pred_scenes)
    )

    for k in [
        "emotion",
        "setting",
        "atmosphere",
        "state",
        "theme"
    ]:

        p,r,f=attr_metrics(
            gt_scenes,
            pred_scenes,
            k
        )

        results[f"{k}_precision"].append(p)
        results[f"{k}_recall"].append(r)
        results[f"{k}_f1"].append(f)

    results["state_transition"].append(
        state_transition(gt_scenes,pred_scenes)
    )

    results["causal"].append(
        causal(gt_scenes,pred_scenes)
    )

    results["schema"].append(
        schema(pred_scenes)
    )


# =========================================================
# AGGREGATE STRUCTURAL
# =========================================================

metrics={
k:float(np.mean(v))
for k,v in results.items()
}


# =========================================================
# TEXT GENERATION QUALITY
# =========================================================

scorer=rouge_scorer.RougeScorer(
['rouge1','rougeL'],
use_stemmer=True
)

r1=[]
rL=[]

for ref,hyp in zip(refs,hyps):

    s=scorer.score(ref,hyp)

    r1.append(s["rouge1"].fmeasure)
    rL.append(s["rougeL"].fmeasure)


bleu=sacrebleu.corpus_bleu(
hyps,
[refs]
).score/100


P,R,F=bertscore(
hyps,
refs,
lang="en",
verbose=False
)

bert_f1=float(F.mean())


embedder=SentenceTransformer(
"all-MiniLM-L6-v2"
)

sim=[]

for a,b in zip(refs,hyps):

    e1=embedder.encode(a)
    e2=embedder.encode(b)

    sim.append(
        cosine_similarity(
            [e1],[e2]
        )[0][0]
    )


# =========================================================
# PRINT RESULTS
# =========================================================

print("\n=========== TEST STRUCTURAL METRICS ===========\n")

for k,v in sorted(metrics.items()):
    print(f"{k}: {v:.4f}")


print("\n=========== TEST QUALITY METRICS ===========\n")

print("Exact Match:",round(exact_match/len(gt_data),4))
print("Structural Match:",round(struct_match/len(gt_data),4))
print("Valid JSON %:",round(valid_json/len(gt_data),4))
print("Avg Scenes/Story:",round(np.mean(scene_counts),4))
print("ROUGE-1:",round(np.mean(r1),4))
print("ROUGE-L:",round(np.mean(rL),4))
print("BLEU:",round(bleu,4))
print("BERTScore:",round(bert_f1,4))
print("Sentence Similarity:",round(np.mean(sim),4))

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.0 MB/s eta 0:00:00
GT: 40
Pred: 40


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


=========== TEST STRUCTURAL METRICS ===========

atmosphere_f1: 0.8735
atmosphere_precision: 0.8504
atmosphere_recall: 0.9145
causal: 0.2927
emotion_f1: 0.7709
emotion_precision: 0.7543
emotion_recall: 0.8162
scene_acc: 0.6752
schema: 1.0000
setting_f1: 0.9402
setting_precision: 0.9359
setting_recall: 0.9487
state_f1: 0.8252
state_precision: 0.7585
state_recall: 0.9145
state_transition: 0.4765
theme_f1: 0.4872
theme_precision: 0.4872
theme_recall: 0.4872

=========== TEST QUALITY METRICS ===========

Exact Match: 0.0
Structural Match: 0.15
Valid JSON %: 0.975
Avg Scenes/Story: 4.0
ROUGE-1: 0.7902
ROUGE-L: 0.7573
BLEU: 0.7069
BERTScore: 0.9662
Sentence Similarity: 0.9789


In [ ]:
# =========================================================
# INSTALL (RUN ONCE)
# =========================================================

!pip -q install rouge-score sacrebleu bert-score sentence-transformers

# =========================================================
# IMPORTS
# =========================================================

import json, ast, re, numpy as np
from collections import defaultdict

from rouge_score import rouge_scorer
import sacrebleu
from bert_score import score as bertscore

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


# =========================================================
# FILES
# =========================================================

GT_PATH="/content/Testing_Ground_Truth_Data.txt"
PRED_PATH="/content/final_test_predictions.json"


# =========================================================
# LOAD DATA
# =========================================================

def load_gt(path):

    with open(path,"r",encoding="utf-8") as f:
        txt=f.read()

    try:
        return json.loads(txt)
    except:
        return ast.literal_eval(txt)


def load_json(path):
    with open(path,"r",encoding="utf-8") as f:
        return json.load(f)


gt_data=load_gt(GT_PATH)
pred_data=load_json(PRED_PATH)

print("GT:",len(gt_data))
print("Pred:",len(pred_data))


# =========================================================
# EXTRACT JSON FROM LLM OUTPUT
# =========================================================

def extract_json(text):

    if text is None:
        return None

    text=str(text)

    text=text.replace("```json","")
    text=text.replace("```","").strip()

    try:
        return json.loads(text)
    except:
        pass

    candidates=re.findall(r"\[.*\]|\{.*\}",text,re.DOTALL)

    for c in reversed(candidates):
        try:
            return json.loads(c)
        except:
            continue

    return None


# =========================================================
# NORMALIZE SCENES
# =========================================================

def normalize(data):

    if isinstance(data,dict):

        if "output" in data:
            data=data["output"]

        elif "scenes" in data:
            data=data["scenes"]

    if isinstance(data,dict):
        data=[data]

    if not isinstance(data,list):
        return []

    scenes=[]

    for s in data:

        if not isinstance(s,dict):
            continue

        scenes.append({
            "emotion":str(s.get("emotion","")).lower().strip(),
            "setting":str(s.get("setting","")).lower().strip(),
            "atmosphere":str(s.get("atmosphere","")).lower().strip(),
            "state":str(s.get("state","")).lower().strip(),
            "theme":str(s.get("theme","")).lower().strip(),
            "state_change":str(s.get("state_change","")).lower().strip(),
            "causal_link":str(s.get("causal_link","")).lower().strip()
        })

    return scenes


# =========================================================
# PRF1
# =========================================================

def prf(gt,pred):

    gt=set([x for x in gt if x!=""])
    pred=set([x for x in pred if x!=""])

    tp=len(gt & pred)
    fp=len(pred-gt)
    fn=len(gt-pred)

    p=tp/(tp+fp+1e-9)
    r=tp/(tp+fn+1e-9)
    f=2*p*r/(p+r+1e-9)

    return p,r,f


def attr_metrics(gt,pred,key):
    return prf(
        [x[key] for x in gt],
        [x[key] for x in pred]
    )


# =========================================================
# STRUCTURAL METRICS
# =========================================================

def scene_acc(gt,pred):
    return 1-abs(len(gt)-len(pred))/max(len(gt),1)


def state_transition_f1(gt,pred):

    a=[x["state_change"] for x in gt if x["state_change"]]
    b=[x["state_change"] for x in pred if x["state_change"]]

    return prf(a,b)[2]


def state_order_consistency(gt,pred):

    a=[x["state"] for x in gt]
    b=[x["state"] for x in pred]

    n=min(len(a),len(b))

    if n==0:
        return 0

    return sum(a[i]==b[i] for i in range(n))/n


def schema(pred):

    req=["emotion","setting","atmosphere","state","theme"]

    total=0
    valid=0

    for s in pred:
        for r in req:
            total+=1
            if s.get(r,"")!="":
                valid+=1

    return valid/(total+1e-9)



# =========================================================
# EMBEDDING MODEL
# =========================================================

embedder=SentenceTransformer(
"all-MiniLM-L6-v2"
)


def causal_link_similarity(gt,pred):

    g=[x["causal_link"] for x in gt if x["causal_link"]]
    p=[x["causal_link"] for x in pred if x["causal_link"]]

    n=min(len(g),len(p))

    if n==0:
        return 0

    sims=[]

    for i in range(n):

        e1=embedder.encode(g[i])
        e2=embedder.encode(p[i])

        sims.append(
            cosine_similarity(
                [e1],[e2]
            )[0][0]
        )

    return float(np.mean(sims))


def causal_composite(gt,pred):

    soc=state_order_consistency(gt,pred)
    trans=state_transition_f1(gt,pred)
    link=causal_link_similarity(gt,pred)

    return (
        .30*soc+
        .30*trans+
        .40*link
    )


# =========================================================
# EVALUATION LOOP
# =========================================================

results=defaultdict(list)

refs=[]
hyps=[]

valid_json=0
exact_match=0
struct_match=0
scene_counts=[]

for gt_story,pred_story in zip(gt_data,pred_data):

    gt_scenes=normalize(gt_story)

    pred_json=extract_json(
        pred_story["predicted_output"]
    )

    if pred_json is None:
        continue

    valid_json+=1

    pred_scenes=normalize(pred_json)

    if len(pred_scenes)==0:
        continue

    scene_counts.append(len(pred_scenes))

    if gt_scenes==pred_scenes:
        exact_match+=1

    if len(gt_scenes)==len(pred_scenes):
        struct_match+=1


    refs.append(json.dumps(gt_scenes))
    hyps.append(json.dumps(pred_scenes))


    results["scene_acc"].append(
        scene_acc(gt_scenes,pred_scenes)
    )

    for k in [
        "emotion",
        "setting",
        "atmosphere",
        "state",
        "theme"
    ]:

        p,r,f=attr_metrics(
            gt_scenes,
            pred_scenes,
            k
        )

        results[f"{k}_precision"].append(p)
        results[f"{k}_recall"].append(r)
        results[f"{k}_f1"].append(f)


    results["state_transition"].append(
        state_transition_f1(
            gt_scenes,
            pred_scenes
        )
    )

    results["causal_composite"].append(
        causal_composite(
            gt_scenes,
            pred_scenes
        )
    )

    results["schema"].append(
        schema(pred_scenes)
    )


metrics={
k:float(np.mean(v))
for k,v in results.items()
}


# =========================================================
# QUALITY METRICS
# =========================================================

scorer=rouge_scorer.RougeScorer(
['rouge1','rougeL'],
use_stemmer=True
)

r1=[]
rL=[]

for ref,hyp in zip(refs,hyps):

    s=scorer.score(ref,hyp)

    r1.append(s["rouge1"].fmeasure)
    rL.append(s["rougeL"].fmeasure)


bleu=sacrebleu.corpus_bleu(
hyps,[refs]
).score/100


P,R,F=bertscore(
hyps,
refs,
lang="en",
verbose=False
)

bert_f1=float(F.mean())


sent_sim=[]

for a,b in zip(refs,hyps):

    e1=embedder.encode(a)
    e2=embedder.encode(b)

    sent_sim.append(
        cosine_similarity(
            [e1],[e2]
        )[0][0]
    )


# =========================================================
# RESULTS
# =========================================================

print("\n===== STRUCTURAL + CAUSAL METRICS =====\n")

for k,v in sorted(metrics.items()):
    print(f"{k}: {v:.4f}")


print("\n===== GENERATION QUALITY =====\n")

print("Exact Match:",round(exact_match/len(gt_data),4))
print("Structural Match:",round(struct_match/len(gt_data),4))
print("Valid JSON %:",round(valid_json/len(gt_data),4))
print("Avg Scenes/Story:",round(np.mean(scene_counts),4))
print("ROUGE-1:",round(np.mean(r1),4))
print("ROUGE-L:",round(np.mean(rL),4))
print("BLEU:",round(bleu,4))
print("BERTScore:",round(bert_f1,4))
print("Sentence Similarity:",round(np.mean(sent_sim),4))

GT: 40
Pred: 40


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



===== STRUCTURAL + CAUSAL METRICS =====

atmosphere_f1: 0.8735
atmosphere_precision: 0.8504
atmosphere_recall: 0.9145
causal_composite: 0.4011
emotion_f1: 0.7709
emotion_precision: 0.7543
emotion_recall: 0.8162
scene_acc: 0.6752
schema: 1.0000
setting_f1: 0.9402
setting_precision: 0.9359
setting_recall: 0.9487
state_f1: 0.8252
state_precision: 0.7585
state_recall: 0.9145
state_transition: 0.4765
theme_f1: 0.4872
theme_precision: 0.4872
theme_recall: 0.4872

===== GENERATION QUALITY =====

Exact Match: 0.0
Structural Match: 0.15
Valid JSON %: 0.975
Avg Scenes/Story: 4.0
ROUGE-1: 0.6545
ROUGE-L: 0.5614
BLEU: 0.618
BERTScore: 0.9381
Sentence Similarity: 0.8964
